In [3]:
#Duplicate Detection


from pyspark.sql import functions as F

df = spark.read.parquet(
    "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
)

total_records = df.count()

unique_records = df.dropDuplicates().count()

duplicate_records = total_records - unique_records

duplicate_percentage = (
    duplicate_records / total_records
) * 100

print("Total records:", total_records)
print("Unique records:", unique_records)
print("Exact duplicate excess records:", duplicate_records)
print("Duplicate percentage:", round(duplicate_percentage, 4))


StatementMeta(, 5e72143a-881a-40e9-b3d5-4c0597c19d78, 5, Finished, Available, Finished, False)

Total records: 3475226
Unique records: 3475226
Exact duplicate excess records: 0
Duplicate percentage: 0.0


In [2]:
#Investigate special taxi zones

from pyspark.sql import functions as F

# 1. Load January 2025 Yellow Taxi dataset
df = spark.read.parquet(
    "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
)

# 2. Load Taxi Zone Lookup dataset
zone_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/reference/taxi_zone_lookup.csv")
)

print("Both datasets loaded successfully!")

# 3. Verify record counts
print("\nTotal taxi records:", df.count())
print("Total taxi zones:", zone_df.count())
print("Unique taxi zone IDs:", zone_df.select("LocationID").distinct().count())

# 4. Investigate special or unknown taxi zones
print("\nSPECIAL OR UNKNOWN TAXI ZONES")

zone_df.filter(
    F.lower(F.col("Zone")).contains("unknown")
    | F.lower(F.col("Borough")).contains("unknown")
    | F.lower(F.col("Zone")).contains("n/a")
).show(truncate=False)

# 5. Analyse taxi pickups by borough
print("\nTAXI PICKUPS BY BOROUGH")

df.join(
    zone_df,
    df.PULocationID == zone_df.LocationID,
    "left"
).groupBy(
    "Borough"
).count().orderBy(
    F.desc("count")
).show(truncate=False)

# 6. Check unmatched pickup locations
print("\nUNMATCHED PICKUP LOCATIONS")

unmatched_pickups = df.join(
    zone_df,
    df.PULocationID == zone_df.LocationID,
    "left_anti"
)

print("Unmatched pickup records:", unmatched_pickups.count())

print("\nGeographic data quality assessment completed!")


StatementMeta(, 5e72143a-881a-40e9-b3d5-4c0597c19d78, 4, Finished, Available, Finished, False)

Both datasets loaded successfully!

Total taxi records: 3475226
Total taxi zones: 265
Unique taxi zone IDs: 265

SPECIAL OR UNKNOWN TAXI ZONES
+----------+-------+----+------------+
|LocationID|Borough|Zone|service_zone|
+----------+-------+----+------------+
|264       |Unknown|N/A |N/A         |
+----------+-------+----+------------+


TAXI PICKUPS BY BOROUGH
+-------------+-------+
|Borough      |count  |
+-------------+-------+
|Manhattan    |3089275|
|Queens       |294986 |
|Brooklyn     |66070  |
|Bronx        |14741  |
|Unknown      |8141   |
|N/A          |1380   |
|EWR          |377    |
|Staten Island|256    |
+-------------+-------+


UNMATCHED PICKUP LOCATIONS
Unmatched pickup records: 0

Geographic data quality assessment completed!


In [4]:
#Check for unmatched pickup zones

#Load your dataset again
df = spark.read.parquet(
    "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
)

print("Dataset loaded successfully!")
unmatched_pickups = (
    df.join(
        zone_df,
        df.PULocationID == zone_df.LocationID,
        "left_anti"
    )
)

print(
    "Unmatched pickup records:",
    unmatched_pickups.count()
)


StatementMeta(, a799e0a9-15b4-4f2d-abbb-d7b864b4619d, 6, Finished, Available, Finished, False)

Dataset loaded successfully!
Unmatched pickup records: 0


In [2]:
#Explore the reference table

print("Total zone lookup records:", zone_df.count())

zone_df.select("LocationID").distinct().count()


StatementMeta(, a799e0a9-15b4-4f2d-abbb-d7b864b4619d, 4, Finished, Available, Finished, False)

Total zone lookup records: 265


265

In [1]:
#Read the lookup table in PySpark
zone_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("Files/reference/taxi_zone_lookup.csv")
)

zone_df.printSchema()
zone_df.show(5, truncate=False)

StatementMeta(, a799e0a9-15b4-4f2d-abbb-d7b864b4619d, 3, Finished, Available, Finished, False)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


In [2]:
#Validate the January Reporting Period
#Load your dataset again
df = spark.read.parquet(
    "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
)

print("Dataset loaded successfully!")

from pyspark.sql import functions as F

df.select(
    F.count(
        F.when(
            F.col("tpep_pickup_datetime") < F.lit("2025-01-01"),
            1
        )
    ).alias("before_january"),

    F.count(
        F.when(
            F.col("tpep_pickup_datetime") >= F.lit("2025-02-01"),
            1
        )
    ).alias("after_january")
).show()


StatementMeta(, ff150275-5379-4c87-b3f4-3be2f9245859, 4, Finished, Available, Finished, False)

Dataset loaded successfully!
+--------------+-------------+
|before_january|after_january|
+--------------+-------------+
|            21|            1|
+--------------+-------------+



In [6]:
#Display five negative-duration trips
df_duration.filter(
    F.col("trip_duration_minutes") < 0
).select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes",
    "PULocationID",
    "DOLocationID",
    "trip_distance"
).show(5, truncate=False)

StatementMeta(, fe4bf74d-6bde-49af-90bf-59a464a4309e, 8, Finished, Available, Finished, False)

+--------------------+---------------------+---------------------+------------+------------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|PULocationID|DOLocationID|trip_distance|
+--------------------+---------------------+---------------------+------------+------------+-------------+
|2025-01-02 12:26:00 |2025-01-02 11:29:58  |-56.03333333333333   |88          |210         |9.0          |
|2025-01-06 16:00:00 |2025-01-06 15:05:30  |-54.5                |208         |254         |3.8          |
|2025-01-15 15:00:00 |2025-01-15 14:42:48  |-17.2                |107         |4           |1.0          |
|2025-01-23 01:44:59 |2024-12-18 07:52:40  |-51472.316666666666  |162         |238         |3.1          |
|2025-01-23 12:30:00 |2025-01-23 11:44:59  |-45.016666666666666  |236         |116         |3.9          |
+--------------------+---------------------+---------------------+------------+------------+-------------+
only showing top 5 rows


In [5]:

from pyspark.sql import functions as F

df_duration = df.withColumn(
    "trip_duration_minutes",
    (
        F.unix_timestamp("tpep_dropoff_datetime")
        - F.unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

df_duration.select(
    F.min("trip_duration_minutes").alias("min_duration"),
    F.max("trip_duration_minutes").alias("max_duration"),
    F.avg("trip_duration_minutes").alias("avg_duration"),
    F.count(
        F.when(F.col("trip_duration_minutes") < 0, 1)
    ).alias("negative_duration"),
    F.count(
        F.when(F.col("trip_duration_minutes") == 0, 1)
    ).alias("zero_duration"),
    F.count(
        F.when(F.col("trip_duration_minutes") > 180, 1)
    ).alias("over_3_hours")
).show(truncate=False)


StatementMeta(, fe4bf74d-6bde-49af-90bf-59a464a4309e, 7, Finished, Available, Finished, False)

+-------------------+-----------------+-----------------+-----------------+-------------+------------+
|min_duration       |max_duration     |avg_duration     |negative_duration|zero_duration|over_3_hours|
+-------------------+-----------------+-----------------+-----------------+-------------+------------+
|-51472.316666666666|5626.316666666667|15.01811561799608|124              |1927         |1377        |
+-------------------+-----------------+-----------------+-----------------+-------------+------------+



In [4]:
#Investigate the extreme values


from pyspark.sql import functions as F

#Load your dataset again
df = spark.read.parquet(
    "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
)

print("Dataset loaded successfully!")

#Display the 10 longest recorded trips
df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount"
).orderBy(
    F.col("trip_distance").desc()
).show(10, truncate=False)

#Compare the mean and median


df.select(
    F.avg("trip_distance").alias("mean_distance"),
    F.percentile_approx(
        "trip_distance", 0.5
    ).alias("median_distance"),
    F.percentile_approx(
        "trip_distance", 0.95
    ).alias("p95_distance"),
    F.percentile_approx(
        "trip_distance", 0.99
    ).alias("p99_distance")
).show(truncate=False)



StatementMeta(, fe4bf74d-6bde-49af-90bf-59a464a4309e, 6, Finished, Available, Finished, False)

Dataset loaded successfully!
+--------------------+---------------------+------------+------------+-------------+-----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|total_amount|
+--------------------+---------------------+------------+------------+-------------+-----------+------------+
|2025-01-22 21:04:00 |2025-01-22 21:13:00  |161         |170         |276423.57    |-4.75      |5.0         |
|2025-01-19 14:51:00 |2025-01-19 14:58:00  |224         |233         |276099.95    |9.13       |13.88       |
|2025-01-21 09:19:00 |2025-01-21 09:45:00  |52          |144         |222167.49    |31.19      |35.94       |
|2025-01-16 10:47:00 |2025-01-16 11:11:00  |263         |48          |206137.99    |24.89      |29.64       |
|2025-01-26 17:46:00 |2025-01-26 17:59:00  |50          |186         |202771.63    |10.1       |14.85       |
|2025-01-14 19:18:00 |2025-01-14 19:36:00  |246         |164         |189687.43    |12.7   

In [6]:
#Data Quality Assessment: Invalid Trip Distances

from pyspark.sql import functions as F

df.select(
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),
    F.avg("trip_distance").alias("avg_distance"),
    F.count(F.when(F.col("trip_distance") < 0, 1)).alias("negative_distance"),
    F.count(F.when(F.col("trip_distance") == 0, 1)).alias("zero_distance"),
    F.count(F.when(F.col("trip_distance") > 100, 1)).alias("over_100_miles")
).show(truncate=False)

StatementMeta(, 4266c6ed-0aeb-4903-872f-506b1246fec7, 8, Finished, Available, Finished, False)

+------------+------------+-----------------+-----------------+-------------+--------------+
|min_distance|max_distance|avg_distance     |negative_distance|zero_distance|over_100_miles|
+------------+------------+-----------------+-----------------+-------------+--------------+
|0.0         |276423.57   |5.855126178843539|0                |90893        |162           |
+------------+------------+-----------------+-----------------+-------------+--------------+



In [5]:
missing_pattern = df.groupBy(
    F.col("passenger_count").isNull().alias("passenger_missing"),
    F.col("RatecodeID").isNull().alias("ratecode_missing"),
    F.col("store_and_fwd_flag").isNull().alias("flag_missing"),
    F.col("congestion_surcharge").isNull().alias("congestion_missing"),
    F.col("Airport_fee").isNull().alias("airport_fee_missing")
).count()

missing_pattern.show(truncate=False)

StatementMeta(, 4266c6ed-0aeb-4903-872f-506b1246fec7, 7, Finished, Available, Finished, False)

+-----------------+----------------+------------+------------------+-------------------+-------+
|passenger_missing|ratecode_missing|flag_missing|congestion_missing|airport_fee_missing|count  |
+-----------------+----------------+------------+------------------+-------------------+-------+
|false            |false           |false       |false             |false              |2935077|
|true             |true            |true        |true              |true               |540149 |
+-----------------+----------------+------------+------------------+-------------------+-------+



In [4]:
#missing-value analysis

from pyspark.sql import functions as F

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show(truncate=False)

StatementMeta(, 4266c6ed-0aeb-4903-872f-506b1246fec7, 6, Finished, Available, Finished, False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|0       |0                   |0                    |540149         |0            |540149    |540149            |0           |0   

In [5]:
#Check your dataset's date range

from pyspark.sql import functions as F

df.select(
    F.min("tpep_pickup_datetime").alias("earliest_pickup"),
    F.max("tpep_pickup_datetime").alias("latest_pickup")
).show(truncate=False)

StatementMeta(, 43415597-7eaa-4e6f-8fb0-12b3437abd13, 7, Finished, Available, Finished, False)

+-------------------+-------------------+
|earliest_pickup    |latest_pickup      |
+-------------------+-------------------+
|2024-12-31 20:47:55|2025-02-01 00:00:44|
+-------------------+-------------------+



In [4]:
#Inspect the first five records

df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "fare_amount"
).show(5, truncate=False)


StatementMeta(, 43415597-7eaa-4e6f-8fb0-12b3437abd13, 6, Finished, Available, Finished, False)

+--------------------+---------------------+------------+------------+---------------+-------------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|
+--------------------+---------------------+------------+------------+---------------+-------------+-----------+
|2025-01-01 00:18:38 |2025-01-01 00:26:59  |229         |237         |1              |1.6          |10.0       |
|2025-01-01 00:32:40 |2025-01-01 00:35:13  |236         |237         |1              |0.5          |5.1        |
|2025-01-01 00:44:04 |2025-01-01 00:46:01  |141         |141         |1              |0.6          |5.1        |
|2025-01-01 00:14:27 |2025-01-01 00:20:01  |244         |244         |3              |0.52         |7.2        |
|2025-01-01 00:21:34 |2025-01-01 00:25:06  |244         |116         |3              |0.66         |5.8        |
+--------------------+---------------------+------------+------------+---------------+----------

In [3]:
#Count the records
total_rows = df.count()

print("Total taxi trip records:", total_rows)

StatementMeta(, 43415597-7eaa-4e6f-8fb0-12b3437abd13, 5, Finished, Available, Finished, False)

Total taxi trip records: 3475226


In [3]:
#Read your Parquet dataset
df = spark.read.parquet(
    "Files/yellow_taxi/yellow_tripdata_2025-01.parquet"
)

df.printSchema()


StatementMeta(, 4266c6ed-0aeb-4903-872f-506b1246fec7, 5, Finished, Available, Finished, False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [2]:
notebookutils.fs.ls("Files/yellow_taxi/")

StatementMeta(, 4266c6ed-0aeb-4903-872f-506b1246fec7, 4, Finished, Available, Finished, False)

[FileInfo(path=abfss://837a4b4c-2337-406e-9aa8-636a9c427790@onelake.dfs.fabric.microsoft.com/80277bca-cfba-4479-a6f4-0929872d57f3/Files/yellow_taxi/yellow_tripdata_2025-01.parquet, name=yellow_tripdata_2025-01.parquet, size=59158238)]